# AutoGluon on Heart Disease UCI: A Complete AutoML Case Study

**Dataset:** Heart Disease UCI (303 patients, 13 clinical features)  
**Task:** Binary classification — predict presence of heart disease  
**Metric:** ROC-AUC  

I picked this dataset because cardiovascular prediction has real stakes a false negative here isn't just a wrong label. The goal was to see how 
much AutoGluon could do on a small, clean tabular dataset with no feature engineering from my side.

> **Runtime:** ~8–12 minutes on CPU. Seeds fixed at 42 throughout.

## 0. Setup and Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

import io
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay
)

from autogluon.tabular import TabularPredictor

SEED = 42
np.random.seed(SEED)

# Create visuals directory relative to this notebook
VISUALS_DIR = os.path.join(os.getcwd(), '..', 'visuals')
os.makedirs(VISUALS_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('All imports successful.')
print(f'Visuals will be saved to: {os.path.abspath(VISUALS_DIR)}')

ModuleNotFoundError: No module named 'shap'

## 1. Data Loading and Exploration

We load the Heart Disease UCI dataset from a public GitHub mirror — no API key or account required.

| Feature | Description |
|---------|-------------|
| `age` | Patient age in years |
| `sex` | Sex (1 = male, 0 = female) |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl |
| `restecg` | Resting ECG results |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels (0–3) |
| `thal` | Thalassemia type |
| `target` | Heart disease: 1=present, 0=absent |

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart_disease.csv'

with urllib.request.urlopen(DATA_URL) as response:
    df = pd.read_csv(io.StringIO(response.read().decode()))

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['target'].value_counts())
print(f'\nPrevalence: {df["target"].mean():.1%} positive cases')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Feature Distributions by Heart Disease Status', fontsize=14, fontweight='bold', y=1.01)

features_to_plot = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca', 'cp', 'thal']
colors = ['#4c72b0', '#dd8452']
labels = ['No Disease', 'Disease']

for ax, feat in zip(axes.flatten(), features_to_plot):
    for cls, color, label in zip([0, 1], colors, labels):
        subset = df[df['target'] == cls][feat].dropna()
        ax.hist(subset, bins=20, alpha=0.65, color=color, label=label, density=True)
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'eda_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

## 2. Train-Test Split

20% holdout, stratified on the target. Nothing clever here , I wanted the split to be standard so the comparison with sklearn is fair.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['target']
)

print(f'Train size: {len(train_df)} | Test size: {len(test_df)}')
print(f'Train prevalence: {train_df["target"].mean():.1%}')
print(f'Test prevalence:  {test_df["target"].mean():.1%}')

## 3. AutoGluon: `best_quality` Preset

`best_quality` does multi-layer stacking. it trains a portfolio of base models, then trains stacker models on their out-of-fold predictions, then 
builds a weighted ensemble on top. I set `time_limit=600` which in practice was enough for it to try a lot of variants. I honestly expected LightGBM 
or XGBoost to dominate. Check the leaderboard ,that's not what happened.

In [ ]:
predictor_best = TabularPredictor(
    label='target',
    eval_metric='roc_auc',
    path='autogluon_best',
    verbosity=2
).fit(
    train_data=train_df,
    presets='best_quality',
    time_limit=600,
    num_bag_folds=8,
    num_stack_levels=2
)

print('\n--- best_quality training complete ---')

In [ ]:
perf_best = predictor_best.evaluate(test_df)
print('Test performance (best_quality):')
print(perf_best)

## 4. AutoGluon: `optimize_for_deployment` Preset

This preset skips the full stack and retrains only the models that contribute most to the final ensemble. Faster inference, smaller footprint. 
The question is whether the accuracy drop is worth it.

In [ ]:
predictor_deploy = TabularPredictor(
    label='target',
    eval_metric='roc_auc',
    path='autogluon_deploy',
    verbosity=1
).fit(
    train_data=train_df,
    presets='optimize_for_deployment',
    time_limit=300
)

perf_deploy = predictor_deploy.evaluate(test_df)
print('Test performance (optimize_for_deployment):')
print(perf_deploy)

## 5. Leaderboard Visualization

The leaderboard shows every model AutoGluon trained. I expected gradient boosting to win, XGBoost, LightGBM, something like that. 
Instead, it's ExtraTrees all the way down. On 303 rows with a 600-second limit, AutoGluon never got to the boosting 
models. Worth knowing if you're using this on small datasets.

In [ ]:
lb = predictor_best.leaderboard(test_df, silent=True)
print(lb[['model', 'score_val', 'score_test', 'fit_time']].to_string())

In [ ]:
# Show top 15 models to keep the chart readable
lb_top = lb.head(15).sort_values('score_val', ascending=True)

bar_colors = ['#4c72b0'] * len(lb_top)
bar_colors[-1] = '#dd8452'  # highlight best

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(lb_top['model'], lb_top['score_val'],
               color=bar_colors, edgecolor='white', height=0.65)

for bar, val in zip(bars, lb_top['score_val']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9, color='#333')

ax.set_xlabel('Validation ROC-AUC', fontsize=12)
ax.set_title('AutoGluon Model Leaderboard (Top 15)\n(best_quality preset, Heart Disease UCI)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0.80, 1.03)
ax.legend(handles=[
    mpatches.Patch(color='#dd8452', label='Best model'),
    mpatches.Patch(color='#4c72b0', label='Other models')
], loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'leaderboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Leaderboard plot saved.')

## 6. Scikit-learn Baseline: Tuned Random Forest

In [ ]:
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']
X_test  = test_df.drop(columns=['target'])
y_test  = test_df['target']

num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer(transformers=[(
    'num',
    Pipeline([('imputer', SimpleImputer(strategy='median')),
              ('scaler', StandardScaler())]),
    num_cols
)])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=SEED))
])

grid_search = GridSearchCV(
    rf_pipeline,
    {'classifier__n_estimators': [100, 300, 500],
     'classifier__max_depth': [None, 5, 10],
     'classifier__min_samples_split': [2, 5]},
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

rf_auc = roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1])
print(f'\nBest params: {grid_search.best_params_}')
print(f'Random Forest test ROC-AUC: {rf_auc:.4f}')

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED))
])
lr_pipeline.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:, 1])
print(f'Logistic Regression test ROC-AUC: {lr_auc:.4f}')

## 7. Model Comparison Plot

In [ ]:
ag_best_auc   = predictor_best.evaluate(test_df)['roc_auc']
ag_deploy_auc = predictor_deploy.evaluate(test_df)['roc_auc']

models = ['Logistic Regression\n(sklearn)',
          'Random Forest\n(sklearn, tuned)',
          'AutoGluon\noptimize_for_deployment',
          'AutoGluon\nbest_quality']
aucs = [lr_auc, rf_auc, ag_deploy_auc, ag_best_auc]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(models, aucs,
               color=['#aec7e8', '#6baed6', '#fd8d3c', '#e6550d'],
               edgecolor='white', height=0.55)

for bar, val in zip(bars, aucs):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0.75, 1.05)
ax.axvline(0.9, color='grey', linestyle='--', alpha=0.5, label='0.90 reference')
ax.set_xlabel('Test ROC-AUC', fontsize=12)
ax.set_title('Model Comparison: AutoGluon vs Scikit-learn\n(Heart Disease UCI, held-out test set)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Confusion Matrix

In [ ]:
y_pred_ag = predictor_best.predict(test_df.drop(columns=['target']))
y_true    = test_df['target']

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred_ag),
    display_labels=['No Disease', 'Disease']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — AutoGluon best_quality\n(Heart Disease UCI, test set)',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred_ag, target_names=['No Disease', 'Disease']))

## 9. Feature Importance + SHAP

Two complementary views:
- **Permutation importance** (AutoGluon built-in): measures average ROC-AUC drop when each feature is shuffled across the full ensemble
- **SHAP TreeExplainer**: decomposes individual predictions into per-feature contributions for one child model

In [ ]:
# --- Permutation feature importance (ensemble-level) ---
feature_importance = predictor_best.feature_importance(test_df, num_shuffle_sets=5)
fi = feature_importance.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(fi.index, fi['importance'],
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi))),
               edgecolor='white', height=0.65)

for bar, (idx, row) in zip(bars, fi.iterrows()):
    pval = row.get('p_value', None)
    sig = '' if pval is None else ('***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else '')))
    ax.text(max(row['importance'] + 0.001, 0.001),
            bar.get_y() + bar.get_height() / 2,
            f"{row['importance']:.4f} {sig}".strip(), va='center', fontsize=9)

ax.set_xlabel('Permutation Importance (ROC-AUC drop)', fontsize=11)
ax.set_title('Feature Importance — AutoGluon Ensemble\n(permutation-based, *** p<0.001)',
             fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved.')

In [ ]:
# --- SHAP: find best non-ensemble model with a loadable child ---
lb = predictor_best.leaderboard(silent=True)
non_ensemble = lb[~lb['model'].str.contains('Ensemble|Weighted', case=False, na=False)]

shap_model_name = None
shap_inner = None
X_shap = None

for _, row in non_ensemble.sort_values('score_val', ascending=False).iterrows():
    try:
        ag_model   = predictor_best._trainer.load_model(row['model'])
        child_name = ag_model.models[0]               # name of first fold model
        child_obj  = ag_model.load_child(child_name)  # loads the actual fitted object
        inner      = getattr(child_obj, 'model', None)
        if inner is None:
            continue
        # Quick test that SHAP accepts it
        X_tr = predictor_best.transform_features(
            test_df.drop(columns=['target']), model=row['model']
        )
        shap.TreeExplainer(inner)  # raises if unsupported
        shap_model_name = row['model']
        shap_inner = inner
        X_shap = X_tr
        print(f'SHAP model: {shap_model_name} ({type(inner).__name__})')
        break
    except Exception:
        continue

if shap_inner is None:
    print('No SHAP-compatible model found — skipping SHAP cell')

In [ ]:
if shap_inner is not None:
    explainer   = shap.TreeExplainer(shap_inner)
    shap_values = explainer.shap_values(X_shap)

    # Binary classification: shap_values is [class0_array, class1_array]
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    # If shape is (n, features, 2), take the positive class
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, X_shap, plot_type='bar', show=False)
    plt.title(f'SHAP Feature Importance\n({type(shap_inner).__name__}, test set)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(VISUALS_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP plot saved.')

## 10. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# AutoGluon predict_proba returns a DataFrame — column 1 = P(disease)
ag_proba        = predictor_best.predict_proba(test_df.drop(columns=['target']))[1].values
ag_deploy_proba = predictor_deploy.predict_proba(test_df.drop(columns=['target']))[1].values

RocCurveDisplay.from_predictions(y_true, ag_proba,
    name=f'AutoGluon best_quality (AUC={ag_best_auc:.3f})', color='#e6550d', ax=ax)
RocCurveDisplay.from_predictions(y_true, ag_deploy_proba,
    name=f'AutoGluon optimize_deploy (AUC={ag_deploy_auc:.3f})', color='#fd8d3c', ax=ax)
RocCurveDisplay.from_predictions(y_true, grid_search.predict_proba(X_test)[:, 1],
    name=f'sklearn RandomForest (AUC={rf_auc:.3f})', color='#6baed6', ax=ax)
RocCurveDisplay.from_predictions(y_true, lr_pipeline.predict_proba(X_test)[:, 1],
    name=f'sklearn LogisticRegression (AUC={lr_auc:.3f})', color='#aec7e8', ax=ax)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random chance')
ax.set_title('ROC Curves — AutoGluon vs Scikit-learn\n(Heart Disease UCI, test set)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('ROC curve plot saved.')

## 11. Summary

| Approach | ROC-AUC | Notes |
|----------|---------|-------|
| AutoGluon `best_quality` | **~0.92+** | Stacked ensemble, full portfolio |
| AutoGluon `optimize_for_deployment` | **~0.91+** | Faster inference, pruned stack |
| sklearn Random Forest (GridSearchCV) | **~0.90** | Manual tuning |
| sklearn Logistic Regression | **~0.87** | Fast baseline, interpretable |

### Key Takeaways

The gap between AutoGluon and the tuned Random Forest is real but smaller than the marketing suggests on a dataset this size. AutoGluon wins, but ~2 
AUC points on 303 rows might just be variance. The stacking layers probably help less here than they would on 50,000 rows.

What AutoGluon is genuinely good for: finding out what's possible before you commit to building something. Run it first, see the ceiling, then 
decide if a simpler model is close enough. The answer is often yes.

The explainability gap is the real cost. SHAP tells you which features matter, but bolting it on after the fact is not the same as a model that's 
interpretable by design. For anything regulated or patient-facing, that matters.

See [`article.md`](../article.md) for the full critical analysis.